# Day 1 · Section 6: Position Information and RoPE

Standalone student notebook. Run cells in order, change the examples, and use the checks to explain what happened. It contains no workshop slides. CPU exercises work without downloads; optional Qwen3 cells require network access and, where indicated, a suitable GPU.


## Goals · 6.1–6.6

Show why token order matters, test permutation equivariance of content-only attention, create a simple explicit position signal, rotate 2D vector pairs and verify that Q–K dot products respond to relative offsets. NumPy and PyTorch paths run independently on CPU.


In [ ]:
import sys, subprocess, numpy as np
try:
    import torch
    DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print('PyTorch',torch.__version__,'device',DEVICE)
except ImportError:
    torch=None;DEVICE=None
    print('PyTorch unavailable locally; NumPy path remains runnable. Colab normally includes PyTorch.')
rng=np.random.default_rng(7)


In [ ]:
def softmax_np(x,axis=-1):
    z=x-np.max(x,axis=axis,keepdims=True);e=np.exp(z);return e/e.sum(axis=axis,keepdims=True)
X=np.array([[1.,0.],[0.,1.],[1.,1.]])
def unmasked_attention(x):
    return softmax_np(x@x.T/np.sqrt(x.shape[-1]))@x
permutation=np.array([2,0,1])
print('original output:\n',np.round(unmasked_attention(X),3))
print('permuted output:\n',np.round(unmasked_attention(X[permutation]),3))
assert np.allclose(unmasked_attention(X[permutation]),unmasked_attention(X)[permutation])
print('Without a position signal or mask, permuting rows permutes outputs the same way.')


### 6.3 · Add a simple position signal

Adding a position vector before attention is one historical design. Qwen3 instead applies rotary position embeddings to Query and Key pairs; the next cells isolate rotation.


In [ ]:
position=np.array([[0.,0.],[.2,0.],[.4,0.]])
print('content only first row:',unmasked_attention(X)[0])
print('content + position first row:',unmasked_attention(X+position)[0])
print('Reordering content while keeping positions fixed changes the output differently:')
print(np.round(unmasked_attention(X[permutation]+position),3))
assert not np.allclose(unmasked_attention(X[permutation]+position),unmasked_attention(X+position)[permutation])


### 6.4–6.5 · Rotate a pair

Position zero is the unrotated reference. For a particular vector pair, position `m` applies angle `m × theta`. Different vector pairs in a real attention head use different frequencies.


In [ ]:
def rotate_np(x,position,theta=np.pi/6):
    angle=position*theta;c,s=np.cos(angle),np.sin(angle)
    R=np.array([[c,-s],[s,c]])
    return R@np.asarray(x,dtype=float)
v=np.array([1.,0.])
for m in range(4):print('position',m,'rotated:',np.round(rotate_np(v,m),3))
assert np.allclose(rotate_np(v,0),v)
if torch is not None:
    def rotate_torch(x,position,theta=np.pi/6):
        angle=position*theta;c,s=np.cos(angle),np.sin(angle)
        R=torch.tensor([[c,-s],[s,c]],dtype=x.dtype,device=x.device)
        return R@x
    v_t=torch.tensor(v,dtype=torch.float32,device=DEVICE)
    assert np.allclose(rotate_torch(v_t,2).cpu().numpy(),rotate_np(v,2),atol=1e-6)


In [ ]:
q=np.array([1.,0.]);k=np.array([.7,.7])
for m,n in [(0,3),(2,5),(3,6),(2,4)]:
    print('m,n:',m,n,'offset:',n-m,'Q·K:',round(rotate_np(q,m)@rotate_np(k,n),5))
assert np.isclose(rotate_np(q,0)@rotate_np(k,3),rotate_np(q,2)@rotate_np(k,5))
assert not np.isclose(rotate_np(q,2)@rotate_np(k,4),rotate_np(q,2)@rotate_np(k,5))


### 6.6 · Apply pairwise rotation in an attention pipeline

The even and odd feature indices form pairs. Rotate Q and K, then calculate attention scores. The Values remain unchanged. This toy uses one frequency per pair.


In [ ]:
Q=np.array([[1.,0.,.8,.2],[.8,.2,.5,.5],[.2,.9,.1,.8]])
K=np.array([[.9,.1,.7,.3],[.1,.9,.6,.4],[.4,.6,.2,.8]])
V=np.eye(3)
def rotate_pairs(row,position):
    return np.concatenate([rotate_np(row[:2],position,np.pi/5),rotate_np(row[2:],position,np.pi/12)])
Qr=np.stack([rotate_pairs(row,m) for m,row in enumerate(Q)])
Kr=np.stack([rotate_pairs(row,m) for m,row in enumerate(K)])
scores=Qr@Kr.T/np.sqrt(Q.shape[-1]);output=softmax_np(scores)@V
print('Q/K/V shapes:',Qr.shape,Kr.shape,V.shape,'output:',output.shape)
assert output.shape==(3,3)
# Exercise: set both frequencies to zero and compare scores.


## Checks

1. What does permutation equivariance say about content-only attention? A causal mask is another source of order constraints.
2. Why do the pairs `(0,3)` and `(2,5)` have the same score in this example?
3. Does rotation change the vector's length? Why are Q and K rotated before their dot product?
